In [1]:
#%pip install playwright
#!python -m playwright install chromium

In [2]:
import sys
import asyncio
from playwright.async_api import async_playwright
import re
from concurrent.futures import ThreadPoolExecutor

In [5]:
async def get_tournament_details(page, url):
    try:
        print(f"\n[INFO] Connecting to: {url}")
        await page.goto(url, wait_until="networkidle")
        
        # Bypass the cookie consent banner if present
        try:
            cookie_button = page.get_by_role("button", name=re.compile(r"Jag godkänner|Spara|Accept|I agree", re.IGNORECASE))
            if await cookie_button.is_visible():
                await cookie_button.click()
                await page.wait_for_timeout(1500)
        except Exception:
            pass
        
        # 1. Extract Tournament Title (Cleaned from browser page title or main header)
        raw_title = await page.title()
        # TournamentSoftware titles usually look like "TournamentTitle - Organization" or "TournamentTitle | Page"
        title = raw_title.split(" - ")[0].split(" | ")[0].strip()
        
        # 2. Extract Last Changed Date from visible text layer
        visible_text = await page.inner_text("body")
        
        # Match English "Last changed" or Swedish "Senast ändrad" and grab the next line
        date_match = re.search(r'(?i)(?:Last\s+changed|Senast\s+ändrad)\s*\n+([^\n\r]+)', visible_text)
        
        extracted_date = "Not Found"
        if date_match:
            extracted_date = date_match.group(1).strip()
        else:
            # Fallback proximity check for dates near the keywords
            fallback_match = re.search(r'(?i)(?:Last\s+changed|Senast\s+ändrad).{1,100}(\b\d{1,2}\s+[A-Za-zåäöÅÄÖ]+\s+\d{4}\b|\b\w+,\s+\d{1,2}\s+[A-Za-zåäöÅÄÖ]+\s+\d{4}\b)', visible_text.replace('\n', ' '))
            if fallback_match:
                extracted_date = fallback_match.group(1).strip()

        print(f"[SUCCESS] Title: {title}")
        print(f"[SUCCESS] Last Changed: {extracted_date}")
        return {"url": url, "title": title, "last_changed": extracted_date}

    except Exception as e:
        print(f"[ERROR] Failed to process URL {url}: {e}")
        return {"url": url, "title": "Error", "last_changed": f"Error: {e}"}

async def track_multiple_tournaments():
    # Add your 5 target tournament URLs to this list
    urls = [
        "https://badmintonsweden.tournamentsoftware.com/tournament/6A02A37E-C5B3-400C-ACA9-13665A4FEA2E",
        "https://badmintonsweden.tournamentsoftware.com/tournament/03E5B27C-15E2-4CEC-9602-5A6FA340B5E7",
        "https://badmintonsweden.tournamentsoftware.com/tournament/370F7F49-148E-45CE-9969-A3ABD75D4DD7",
        "https://badmintonsweden.tournamentsoftware.com/tournament/C81237FE-23E8-408D-920F-61855488B9CF",
        "https://badmintonsweden.tournamentsoftware.com/tournament/6BF5703C-1A05-49AA-AB00-FB43C0DE5041"
    ]
    
    async with async_playwright() as p:
        print("[INFO] Launching automated browser...")
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        results = []
        for url in urls:
            if "tournamentsoftware.com" in url:
                data = await get_tournament_details(page, url)
                results.append(data)
        
        await browser.close()
        
        # Final Summary Output Block
        print("\n" + "="*50)
        print("          TOURNAMENT TRACKING SUMMARY          ")
        print("="*50)
        for item in results:
            print(f"🏆 Tournament : {item['title']}")
            print(f"📅 Last Changed: {item['last_changed']}")
            print(f"🔗 Link        : {item['url']}")
            print("-" * 50)

# Run the multi-tracker function inside the Jupyter environment loop
def run_async_in_jupyter(coro):
    def runner():
        if sys.platform == "win32":
            return asyncio.run(coro, loop_factory=asyncio.ProactorEventLoop)
        return asyncio.run(coro)

    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(runner).result()

# Replace `await track_multiple_tournaments()` with:
run_async_in_jupyter(track_multiple_tournaments())

[INFO] Launching automated browser...

[INFO] Connecting to: https://badmintonsweden.tournamentsoftware.com/tournament/6A02A37E-C5B3-400C-ACA9-13665A4FEA2E
[SUCCESS] Title: Norrköpings Singel Cup 2026
[SUCCESS] Last Changed: den 16 november 2026 10:29 (GMT +01:00)

[INFO] Connecting to: https://badmintonsweden.tournamentsoftware.com/tournament/03E5B27C-15E2-4CEC-9602-5A6FA340B5E7
[SUCCESS] Title: Arosträffen 2026 sponserd by Aimsoft och Bellevue
[SUCCESS] Last Changed: den 15 november 2026 19:34 (GMT +01:00)

[INFO] Connecting to: https://badmintonsweden.tournamentsoftware.com/tournament/370F7F49-148E-45CE-9969-A3ABD75D4DD7
[SUCCESS] Title: Stålfjädern 2026
[SUCCESS] Last Changed: den 15 november 2026 20:09 (GMT +01:00)

[INFO] Connecting to: https://badmintonsweden.tournamentsoftware.com/tournament/C81237FE-23E8-408D-920F-61855488B9CF
[SUCCESS] Title: BSO 2026 Täby U13 SJT
[SUCCESS] Last Changed: den 26 oktober 2026 13:26 (GMT +01:00)

[INFO] Connecting to: https://badmintonsweden.tou